<a href="https://colab.research.google.com/github/Nithu006/J26-IT-388/blob/main/Reserch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("/addiction_2019_features_tfidf_256.csv")

print(df.shape)                 # rows, columns
print(list(df.columns[:15]))    # mudhal 15 column names

keys = ["author", "user", "date", "time", "utc", "subreddit", "comment", "score"]
print([c for c in df.columns if any(k in c.lower() for k in keys)])

df.iloc[:3, :8]

(1257, 350)
['subreddit', 'author', 'date', 'post', 'automated_readability_index', 'coleman_liau_index', 'flesch_kincaid_grade_level', 'flesch_reading_ease', 'gulpease_index', 'gunning_fog_index', 'lix', 'smog_index', 'wiener_sachtextformel', 'n_chars', 'n_long_words']
['subreddit', 'author', 'date', 'liwc_time', 'tfidf_time']


,subreddit,author,date,post,automated_readability_index,coleman_liau_index,flesch_kincaid_grade_level,flesch_reading_ease
0,addiction,Lillie1990,2019/01/01,Quitting coke and need advice So I’m 20. I’ve ...,2.315272,4.091255,3.580870,89.872283
1,addiction,P3nw00dx,2019/01/01,I am addicted to sleep I really really like to...,1.591231,2.547960,3.821538,90.818462
2,addiction,throwaway146900,2019/01/01,Addicted to scratching my balls This may sound...,2.829911,4.762222,3.912143,87.370000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import pandas as pd

users = pd.read_csv("/content/users_annonym.csv")
tl = pd.read_csv("/content/timelines_annonym.csv")

print("USERS:", users.shape)
print(users.columns.tolist())
print(users.head(3))

print("TIMELINES:", tl.shape)
print(tl.columns.tolist())
print(tl.head(3))

FileNotFoundError: [Errno 2] No such file or directory: '/content/users_annonym.csv'

In [2]:
import pandas as pd

users = pd.read_csv("/users_annonym.csv")
tl = pd.read_csv("/timelines_annonym.csv")

print("USERS:", users.shape)
print(users.columns.tolist())
print(users.head(3))

print("TIMELINES:", tl.shape)
print(tl.columns.tolist())
print(tl.head(3))

USERS: (165, 6)
['Unnamed: 0', 'participant_id', 'gold_depression', 'gold_anxiety', 'depression', 'anxiety']
   Unnamed: 0 participant_id gold_depression gold_anxiety  depression  anxiety
0           0  participant_0       depressed      anxious           1        1
1           1  participant_1       depressed         none           1        0
2           2  participant_2            none         none           0        0
TIMELINES: (6420, 30)
['Unnamed: 0', 'participant_id', 'created_date', 'anonymised_text', 'admiration', 'amusement', 'anger', 'annoyance', 'anticipation', 'approval', 'confusion', 'curiosity', 'desire', 'disapproval', 'disgust', 'disappointed', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realisation', 'remorse', 'sadness', 'surprise', 'trust']
   Unnamed: 0 participant_id               created_date  \
0           0  participant_0  2023-03-09 05:17:54+00:00   
1           1  participant_0  2023-04-20 04:24:13+01:00   


In [4]:
USER_COL = "participant_id"
TIME_COL = "created_date"
LABEL_COL = "depression"

users = users.drop(columns=["Unnamed: 0"], errors="ignore")
tl = tl.drop(columns=["Unnamed: 0"], errors="ignore")

print("Before:", tl.shape)

tl = tl.drop_duplicates()
tl = tl.dropna(subset=[USER_COL, TIME_COL])
tl[TIME_COL] = pd.to_datetime(tl[TIME_COL], errors="coerce", utc=True)   # mixed +00:00/+01:00 handle aagum
tl = tl.dropna(subset=[TIME_COL])
tl = tl.sort_values([USER_COL, TIME_COL]).reset_index(drop=True)

print("After:", tl.shape)
print("Date range:", tl[TIME_COL].min(), "to", tl[TIME_COL].max())
print("Users:", tl[USER_COL].nunique())

Before: (6420, 29)
After: (6420, 29)
Date range: 2017-09-27 04:39:55+00:00 to 2023-12-09 12:20:25+00:00
Users: 165


In [5]:
counts = tl[USER_COL].value_counts()
print(counts.describe())

span = tl.groupby(USER_COL)[TIME_COL].agg(lambda s: (s.max() - s.min()).days)
print(span.describe())     # oru user-oda tweets ethanai naal span


count    165.000000
mean      38.909091
std      116.233565
min        1.000000
25%        1.000000
50%        5.000000
75%       21.000000
max      940.000000
Name: count, dtype: float64
count     165.000000
mean       87.321212
std       255.716497
min         0.000000
25%         0.000000
50%        17.000000
75%        70.000000
max      2256.000000
Name: created_date, dtype: float64


In [6]:
df = tl.merge(users[[USER_COL, "depression", "anxiety"]], on=USER_COL, how="inner")

print("Users:", df[USER_COL].nunique())
print("Total posts:", len(df))
print("Median posts/user:", df.groupby(USER_COL).size().median())
print(df.drop_duplicates(USER_COL)["depression"].value_counts())

Users: 165
Total posts: 6420
Median posts/user: 5.0
depression
0    105
1     60
Name: count, dtype: int64


In [7]:
df = tl.merge(users[[USER_COL, "depression", "anxiety"]], on=USER_COL, how="inner")

print("Users:", df[USER_COL].nunique(), "| Posts:", len(df))
print(df.drop_duplicates(USER_COL)["depression"].value_counts())

per_user = df.groupby(USER_COL).agg(n=(TIME_COL, "count"), label=("depression", "first"))

print("\nMin tweets -> users kept, depression label split")
for m in [3, 5, 10, 20, 30]:
    sub = per_user[per_user["n"] >= m]
    print(m, "->", len(sub), sub["label"].value_counts().to_dict())

Users: 165 | Posts: 6420
depression
0    105
1     60
Name: count, dtype: int64

Min tweets -> users kept, depression label split
3 -> 107 {0: 62, 1: 45}
5 -> 86 {0: 51, 1: 35}
10 -> 67 {0: 40, 1: 27}
20 -> 48 {0: 26, 1: 22}
30 -> 29 {1: 16, 0: 13}


In [8]:
import numpy as np

MIN_POSTS = 20      # user-ku kuraindha tweets
MAX_POSTS = 60      # oru user-ku mudhal 60 tweets mattum (940 tweets user dominate aagaama)
W = 5               # 5 tweets = 1 time step

cnt = df.groupby(USER_COL).size()
keep = cnt[cnt >= MIN_POSTS].index
d = df[df[USER_COL].isin(keep)].sort_values([USER_COL, TIME_COL])
d = d.groupby(USER_COL).head(MAX_POSTS).copy()

txt = d["anonymised_text"].fillna("")
d["hour"] = d[TIME_COL].dt.hour
d["night"] = d["hour"].between(0, 4).astype(int)            # 12am-5am
d["text_len"] = txt.str.len()
d["mentions"] = txt.str.count(r"\(user-\d+\)")              # social interaction proxy
d["gap_h"] = d.groupby(USER_COL)[TIME_COL].diff().dt.total_seconds() / 3600
d["gap_h"] = d["gap_h"].fillna(0)
d["idx"] = d.groupby(USER_COL).cumcount()
d["win"] = d["idx"] // W

win = d.groupby([USER_COL, "win"]).agg(
    gap_mean=("gap_h", "mean"),
    gap_std=("gap_h", "std"),
    night_ratio=("night", "mean"),
    text_len=("text_len", "mean"),
    mentions=("mentions", "mean"),
    n=("idx", "count"),
).reset_index()

win = win[win["n"] == W].drop(columns="n").fillna(0)        # full windows mattum
win["gap_mean"] = np.log1p(win["gap_mean"])                 # gaps romba skewed, log
win["gap_std"] = np.log1p(win["gap_std"])

print("Users:", win[USER_COL].nunique())
print(win.groupby(USER_COL).size().describe())              # ovvoru user-ku ethanai steps
win.head()

Users: 48
count    48.000000
mean      8.208333
std       3.655124
min       4.000000
25%       4.000000
50%       7.500000
75%      12.000000
max      12.000000
dtype: float64


,participant_id,win,gap_mean,gap_std,night_ratio,text_len,mentions
0,participant_0,0,1.311077,1.823539,0.2,55.0,1.0
1,participant_0,1,1.229722,1.536839,0.2,53.6,0.8
2,participant_0,2,1.521068,1.683549,0.2,65.0,0.4
3,participant_0,3,0.632128,0.815920,0.0,76.8,1.0
4,participant_0,4,0.157336,0.186108,0.0,75.2,0.8


In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

FEATS = ["gap_mean", "gap_std", "night_ratio", "text_len", "mentions"]
lab = users.set_index(USER_COL)["depression"]

uids = sorted(win[USER_COL].unique())
y = np.array([lab[u] for u in uids])
print("Users:", len(y), "| depressed:", y.sum(), "| not:", (y == 0).sum())

# Baseline input: ovvoru user-ku features-oda mean + std (time order illa)
agg = win.groupby(USER_COL)[FEATS].agg(["mean", "std"]).fillna(0)
X_base = agg.loc[uids].values

# LSTM input: ovvoru user-ku 12 steps (kammi-na 0 padding)
T = 12
X_seq = np.zeros((len(uids), T, len(FEATS)), dtype="float32")
mask = np.zeros((len(uids), T), dtype=bool)
for i, u in enumerate(uids):
    s = win[win[USER_COL] == u].sort_values("win")[FEATS].values
    X_seq[i, :len(s)] = s
    mask[i, :len(s)] = True
print(X_base.shape, X_seq.shape)


Users: 48 | depressed: 22 | not: 26
(48, 10) (48, 12, 5)


In [10]:
def metrics(yt, yp, pr):
    return [accuracy_score(yt, yp), precision_score(yt, yp, zero_division=0),
            recall_score(yt, yp), f1_score(yt, yp), roc_auc_score(yt, pr)]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
res = {"LogReg": [], "RandomForest": []}

for tr, te in skf.split(X_base, y):
    sc = StandardScaler().fit(X_base[tr])          # scaler train-la mattum fit
    Xtr, Xte = sc.transform(X_base[tr]), sc.transform(X_base[te])
    models_ = {
        "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "RandomForest": RandomForestClassifier(300, class_weight="balanced", random_state=42),
    }
    for name, m in models_.items():
        m.fit(Xtr, y[tr])
        res[name].append(metrics(y[te], m.predict(Xte), m.predict_proba(Xte)[:, 1]))

print("columns: accuracy, precision, recall, f1, roc_auc")
for k, v in res.items():
    print(k, "mean:", np.round(np.mean(v, 0), 3), "| std:", np.round(np.std(v, 0), 3))

columns: accuracy, precision, recall, f1, roc_auc
LogReg mean: [0.631 0.629 0.74  0.657 0.755] | std: [0.144 0.195 0.146 0.107 0.133]
RandomForest mean: [0.584 0.567 0.56  0.543 0.681] | std: [0.055 0.082 0.159 0.061 0.064]


In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build():
    m = models.Sequential([
        layers.Input((T, len(FEATS))),
        layers.Masking(mask_value=0.0),     # padding steps ignore pannum
        layers.LSTM(16),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy")
    return m

res_lstm = []
for tr, te in skf.split(X_seq, y):
    sc = StandardScaler().fit(X_seq[tr][mask[tr]])      # real steps-la mattum

    def prep(idx):
        Xs = X_seq[idx].copy()
        Xs[mask[idx]] = sc.transform(Xs[mask[idx]])
        Xs[~mask[idx]] = 0
        return Xs

    tf.keras.utils.set_random_seed(42)
    m = build()
    cw = {0: 1.0, 1: (y[tr] == 0).sum() / (y[tr] == 1).sum()}
    m.fit(prep(tr), y[tr], epochs=60, batch_size=8, class_weight=cw, verbose=0)
    pr = m.predict(prep(te), verbose=0).ravel()
    res_lstm.append(metrics(y[te], (pr >= 0.5).astype(int), pr))

print("LSTM mean:", np.round(np.mean(res_lstm, 0), 3), "| std:", np.round(np.std(res_lstm, 0), 3))

LSTM mean: [0.604 0.564 0.7   0.601 0.745] | std: [0.073 0.097 0.253 0.109 0.149]


In [12]:
cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
rows = {k: np.mean(v, 0) for k, v in res.items()}
rows["LSTM"] = np.mean(res_lstm, 0)
table = pd.DataFrame(rows, index=cols).T.round(3)
print(table)
table.to_csv("/content/results_table.csv")


              accuracy  precision  recall     f1  roc_auc
LogReg           0.631      0.629    0.74  0.657    0.755
RandomForest     0.584      0.567    0.56  0.543    0.681
LSTM             0.604      0.564    0.70  0.601    0.745


In [1]:
from sklearn.model_selection import RepeatedStratifiedKFold

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
R = {"LogReg": [], "RandomForest": [], "LSTM": []}

for tr, te in rskf.split(X_base, y):
    sc = StandardScaler().fit(X_base[tr])
    Xtr, Xte = sc.transform(X_base[tr]), sc.transform(X_base[te])

    lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, y[tr])
    R["LogReg"].append(metrics(y[te], lr.predict(Xte), lr.predict_proba(Xte)[:, 1]))

    rf = RandomForestClassifier(300, class_weight="balanced", random_state=42).fit(Xtr, y[tr])
    R["RandomForest"].append(metrics(y[te], rf.predict(Xte), rf.predict_proba(Xte)[:, 1]))

    sc2 = StandardScaler().fit(X_seq[tr][mask[tr]])
    def prep(idx):
        Xs = X_seq[idx].copy()
        Xs[mask[idx]] = sc2.transform(Xs[mask[idx]])
        Xs[~mask[idx]] = 0
        return Xs

    tf.keras.utils.set_random_seed(42)
    m = build()
    cw = {0: 1.0, 1: (y[tr] == 0).sum() / (y[tr] == 1).sum()}
    m.fit(prep(tr), y[tr], epochs=60, batch_size=8, class_weight=cw, verbose=0)
    pr = m.predict(prep(te), verbose=0).ravel()
    R["LSTM"].append(metrics(y[te], (pr >= 0.5).astype(int), pr))

cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
table = pd.DataFrame({k: np.mean(v, 0) for k, v in R.items()}, index=cols).T.round(3)
print(table)
table.to_csv("/content/results_table_repeated.csv")

NameError: name 'X_base' is not defined

In [2]:
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ---------- 1. Load ----------
def find(name):
    hits = glob.glob(f"/content/**/{name}", recursive=True) + glob.glob(f"/{name}")
    assert hits, f"{name} kidaikkala. Files-ah thirumba upload pannunga."
    return hits[0]

users = pd.read_csv(find("users_annonym.csv"))
tl = pd.read_csv(find("timelines_annonym.csv"))

USER_COL, TIME_COL = "participant_id", "created_date"

# ---------- 2. Clean ----------
users = users.drop(columns=["Unnamed: 0"], errors="ignore")
tl = tl.drop(columns=["Unnamed: 0"], errors="ignore")
tl = tl.drop_duplicates().dropna(subset=[USER_COL, TIME_COL])
tl[TIME_COL] = pd.to_datetime(tl[TIME_COL], errors="coerce", utc=True)
tl = tl.dropna(subset=[TIME_COL]).sort_values([USER_COL, TIME_COL]).reset_index(drop=True)
df = tl.merge(users[[USER_COL, "depression", "anxiety"]], on=USER_COL, how="inner")

# ---------- 3. Window features ----------
MIN_POSTS, MAX_POSTS, W = 20, 60, 5
cnt = df.groupby(USER_COL).size()
keep = cnt[cnt >= MIN_POSTS].index
d = df[df[USER_COL].isin(keep)].sort_values([USER_COL, TIME_COL])
d = d.groupby(USER_COL).head(MAX_POSTS).copy()

txt = d["anonymised_text"].fillna("")
d["hour"] = d[TIME_COL].dt.hour
d["night"] = d["hour"].between(0, 4).astype(int)
d["text_len"] = txt.str.len()
d["mentions"] = txt.str.count(r"\(user-\d+\)")
d["gap_h"] = d.groupby(USER_COL)[TIME_COL].diff().dt.total_seconds() / 3600
d["gap_h"] = d["gap_h"].fillna(0)
d["idx"] = d.groupby(USER_COL).cumcount()
d["win"] = d["idx"] // W

win = d.groupby([USER_COL, "win"]).agg(
    gap_mean=("gap_h", "mean"), gap_std=("gap_h", "std"),
    night_ratio=("night", "mean"), text_len=("text_len", "mean"),
    mentions=("mentions", "mean"), n=("idx", "count"),
).reset_index()
win = win[win["n"] == W].drop(columns="n").fillna(0)
win["gap_mean"] = np.log1p(win["gap_mean"])
win["gap_std"] = np.log1p(win["gap_std"])

# ---------- 4. Model inputs ----------
FEATS = ["gap_mean", "gap_std", "night_ratio", "text_len", "mentions"]
lab = users.set_index(USER_COL)["depression"]
uids = sorted(win[USER_COL].unique())
y = np.array([lab[u] for u in uids])
print("Users:", len(y), "| depressed:", y.sum(), "| not:", (y == 0).sum())

agg = win.groupby(USER_COL)[FEATS].agg(["mean", "std"]).fillna(0)
X_base = agg.loc[uids].values

T = 12
X_seq = np.zeros((len(uids), T, len(FEATS)), dtype="float32")
mask = np.zeros((len(uids), T), dtype=bool)
for i, u in enumerate(uids):
    s = win[win[USER_COL] == u].sort_values("win")[FEATS].values
    X_seq[i, :len(s)] = s
    mask[i, :len(s)] = True

# ---------- 5. Helpers ----------
def metrics(yt, yp, pr):
    return [accuracy_score(yt, yp), precision_score(yt, yp, zero_division=0),
            recall_score(yt, yp), f1_score(yt, yp), roc_auc_score(yt, pr)]

def build():
    m = models.Sequential([
        layers.Input((T, len(FEATS))),
        layers.Masking(mask_value=0.0),
        layers.LSTM(16),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy")
    return m

# ---------- 6. Repeated CV (5 folds x 5 repeats) ----------
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
R = {"LogReg": [], "RandomForest": [], "LSTM": []}

for tr, te in rskf.split(X_base, y):
    sc = StandardScaler().fit(X_base[tr])
    Xtr, Xte = sc.transform(X_base[tr]), sc.transform(X_base[te])

    lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, y[tr])
    R["LogReg"].append(metrics(y[te], lr.predict(Xte), lr.predict_proba(Xte)[:, 1]))

    rf = RandomForestClassifier(300, class_weight="balanced", random_state=42).fit(Xtr, y[tr])
    R["RandomForest"].append(metrics(y[te], rf.predict(Xte), rf.predict_proba(Xte)[:, 1]))

    sc2 = StandardScaler().fit(X_seq[tr][mask[tr]])
    def prep(idx):
        Xs = X_seq[idx].copy()
        Xs[mask[idx]] = sc2.transform(Xs[mask[idx]])
        Xs[~mask[idx]] = 0
        return Xs

    tf.keras.utils.set_random_seed(42)
    m = build()
    cw = {0: 1.0, 1: (y[tr] == 0).sum() / (y[tr] == 1).sum()}
    m.fit(prep(tr), y[tr], epochs=60, batch_size=8, class_weight=cw, verbose=0)
    pr = m.predict(prep(te), verbose=0).ravel()
    R["LSTM"].append(metrics(y[te], (pr >= 0.5).astype(int), pr))

cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
table = pd.DataFrame({k: np.mean(v, 0) for k, v in R.items()}, index=cols).T.round(3)
print(table)
table.to_csv("/content/results_table_repeated.csv")


Users: 48 | depressed: 22 | not: 26


              accuracy  precision  recall     f1  roc_auc
LogReg           0.602      0.594   0.592  0.561    0.703
RandomForest     0.593      0.579   0.506  0.509    0.714
LSTM             0.671      0.630   0.742  0.663    0.766


In [ ]:
df["date"] = pd.to_datetime(df["date"])

print("Authors:", df["author"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

counts = df["author"].value_counts()
print("Authors with 2+ posts:", (counts >= 2).sum())
print("Authors with 5+ posts:", (counts >= 5).sum())
print(counts.head(10))


Authors: 1257
Date range: 2019-01-01 00:00:00 to 2019-04-20 00:00:00
Authors with 2+ posts: 0
Authors with 5+ posts: 0
author
LET_AFRICA_DIE       1
Lillie1990           1
P3nw00dx             1
throwaway146900      1
deeseypieces14       1
asdfderp1009         1
wizardslashmagi      1
fuckingburner1232    1
Girl-Here            1
ReallybigTomatos     1
Name: count, dtype: int64


In [ ]:
!wget -O ic_data.zip "https://zenodo.org/records/5716441/files/ic_data.zip?download=1"

--2026-10-19 08:42:13--  https://zenodo.org/records/5716441/files/ic_data.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.63, 188.185.42.113, 188.185.40.132, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.63|:443... connected.
HTTP request sent, awaiting response... 404 NOT FOUND
2026-10-19 08:42:14 ERROR 404: NOT FOUND.

